# Fine-Tune Gemma 4 E2B for Text-to-SQL with Open-RL

This notebook mirrors the current working **Gemma 4 E2B** recipe in this repo:

- base model: `google/gemma-4-e2b`
- tokenizer: the stock `E2B` tokenizer
- prompt format: **plain SQL completion**
- backend: Open-RL single-process engine mode

The existing `texttosql_sft_notebook.ipynb` stays as the Gemma 3 / chat-template walkthrough. This notebook is the Gemma 4 plain-completion path that matches the current `gemma4_e2b` preset in `client/texttosql_sft.py`.

In [ ]:
import asyncio
import json
import logging
import os
import random
import re
import sqlite3
import time
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, cast

import matplotlib.pyplot as plt
import pandas as pd
import tinker
from datasets import load_dataset
from IPython.display import display
from tinker import types
from tqdm.auto import tqdm
from transformers import AutoTokenizer

os.environ.setdefault("TINKER_API_KEY", "tml-dummy-key")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
logging.getLogger("tinker").setLevel(logging.WARNING)

DEFAULT_BASE_URL = "http://127.0.0.1:9003"
DATASET_NAME = "philschmid/gretel-synthetic-text-to-sql"
MAX_SEQ_LENGTH = 512

PLAIN_SQL_PROMPT = """Return only one SQLite query.

Schema:
{context}

Question:
{question}

SQL:
"""


## Config

These values match the current `gemma4_e2b` preset in `client/texttosql_sft.py`.


In [ ]:
@dataclass
class Config:
    base_model: str = "google/gemma-4-e2b"
    tokenizer_name: str = "google/gemma-4-e2b"
    base_url: str = os.getenv("TINKER_BASE_URL") or os.getenv("OPEN_RL_BASE_URL") or DEFAULT_BASE_URL
    steps: int = 100
    batch_size: int = 1
    lora_rank: int = 32
    learning_rate: float = 5e-5
    grad_clip_norm: float = 0.3
    eval_every: int = 100
    train_limit: int = 100
    eval_limit: int = 25
    eval_max_tokens: int = 64
    seed: int = 30
    metrics_path: str = "artifacts/texttosql_gemma4_plain_notebook_metrics.jsonl"
    plot_path: str = "artifacts/texttosql_gemma4_plain_notebook_results.png"
    run_label: str = "Gemma 4 E2B Plain SQL SFT"

config = Config()
display(pd.DataFrame.from_dict(asdict(config), orient="index", columns=["value"]))

## Prerequisites

Start a local Open-RL server in another terminal. With the current server defaults, this is enough:

```bash
cd server
OPEN_RL_BASE_MODEL=google/gemma-4-e2b uv run uvicorn src.main:app --host 127.0.0.1 --port 9003
```

That will infer **single-process engine mode** automatically.

You also need Hugging Face access to `google/gemma-4-e2b`.

In [ ]:
dataset = load_dataset(DATASET_NAME, split="train").shuffle(seed=config.seed)
dataset = dataset.select(range(min(12_500, len(dataset))))
split = dataset.train_test_split(test_size=2_500, shuffle=False)

print(f"Dataset loaded: {len(dataset):,} rows")
print(f"  Training split: {len(split['train']):,}")
print(f"  Eval split:     {len(split['test']):,}")

preview_rows = [split["train"][i] for i in range(3)]
preview_df = pd.DataFrame([
    {
        "question": row["sql_prompt"],
        "schema": row["sql_context"],
        "target_sql": row["sql"],
    }
    for row in preview_rows
])
display(preview_df)

print("Prompt template used for training and inference:\n")
print(PLAIN_SQL_PROMPT.format(context="CREATE TABLE ...", question="How many rows are there?"))


## Connect to Open-RL

We create a LoRA adapter on the server, then use named weight snapshots for sampling and evaluation. This notebook uses the same training settings as the script path:

- `train_attn=True`
- `train_mlp=True`
- `train_unembed=False`


In [ ]:
service_client = tinker.ServiceClient(
    api_key=os.getenv("TINKER_API_KEY", "tml-dummy-key"),
    base_url=config.base_url,
)

try:
    capabilities = await service_client.get_server_capabilities_async()
    server_model = next(
        (getattr(model, "model_name", None) for model in capabilities.supported_models if getattr(model, "model_name", None)),
        None,
    )
except Exception as exc:
    raise RuntimeError(f"Open-RL server at {config.base_url} is not reachable. Start it first.") from exc

if server_model is None:
    raise RuntimeError(f"Server at {config.base_url} is reachable but no model is loaded.")

print(f"Connected to {config.base_url}")
print(f"Server model: {server_model}")

training_client = await service_client.create_lora_training_client_async(
    base_model=config.base_model,
    rank=config.lora_rank,
    seed=config.seed,
    train_mlp=True,
    train_attn=True,
    train_unembed=False,
)
print(f"LoRA adapter created | rank={config.lora_rank} | base={config.base_model}")

tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


## Snapshot and sample the untrained adapter

Because this is the raw `E2B` checkpoint, the baseline usually looks like continuation-style garbage or malformed SQL. The improvement comes from teaching it to continue after a plain `SQL:` prefix.


In [ ]:
sample_row = split["test"][0]
sample_prompt = PLAIN_SQL_PROMPT.format(
    question=sample_row["sql_prompt"],
    context=sample_row["sql_context"],
)
sample_token_ids = tokenizer.encode(sample_prompt, add_special_tokens=False)

weights_path = training_client.save_weights_for_sampler(name="pre_training").result().path
sampler = service_client.create_sampling_client(weights_path)
result = sampler.sample(
    prompt=types.ModelInput.from_ints(tokens=sample_token_ids),
    num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=config.eval_max_tokens, temperature=0.0),
).result()

raw_output = tokenizer.decode(
    result.sequences[0].tokens if result.sequences else [],
    skip_special_tokens=True,
).strip()

print(f"Question: {sample_row['sql_prompt']}")
print(f"Expected SQL: {sample_row['sql']}")
print(f"Model output: {raw_output}")


## Helper functions

These are pulled directly from the current script path so the notebook matches the real training recipe.


In [ ]:
def render_training_texts(question: str, context: str, target_sql: str) -> tuple[str, str]:
    prompt_text = PLAIN_SQL_PROMPT.format(question=question, context=context)
    return prompt_text, prompt_text + target_sql


def clean_sql_for_execution(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    text = text.replace("<|im_start|>", " ").replace("<|im_end|>", " ")
    text = text.strip()
    text = re.sub(r"^assistant\s*[:\-]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^sql\s*[:\-]?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```(?:sql)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    sql_start = re.search(r"\b(with|select|insert|update|delete)\b", text, flags=re.IGNORECASE)
    if sql_start:
        text = text[sql_start.start():]
        statement = re.search(r".*?(?:;|$)", text, flags=re.DOTALL)
        if statement:
            text = statement.group(0)
    return text.strip()


def normalize_sql(text: str) -> str:
    text = clean_sql_for_execution(text)
    text = " ".join(text.split()).lower()
    text = re.sub(r";+\s*$", "", text)
    text = re.sub(r"\s+([,;()])", r"\1", text)
    text = re.sub(r"([,(])\s+", r"\1", text)
    return text


def run_sql(context: str, query: str) -> tuple[list[tuple[Any, ...]] | None, str | None]:
    connection = sqlite3.connect(":memory:")
    try:
        deadline = time.monotonic() + 0.25
        connection.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, 10_000)
        connection.executescript(context)
        rows = connection.execute(query).fetchall()
        normalized_rows = [
            tuple(round(value, 8) if isinstance(value, float) else value for value in row)
            for row in rows
        ]
        return normalized_rows, None
    except sqlite3.Error as exc:
        return None, str(exc)
    finally:
        connection.close()


def sql_results_match(context: str, predicted_sql: str, target_sql: str, target_rows=None):
    predicted_rows, error = run_sql(context, predicted_sql)
    if error is not None:
        return False, f"predicted query error: {error}"

    if target_rows is None:
        target_rows, error = run_sql(context, target_sql)
        if error is not None:
            return False, f"target query error: {error}"

    order_sensitive = any(
        token in f" {normalize_sql(predicted_sql)} {normalize_sql(target_sql)} "
        for token in (" order by ", " limit ", " offset ")
    )
    if not order_sensitive:
        predicted_rows = sorted(predicted_rows, key=repr)
        target_rows = sorted(target_rows, key=repr)
    return predicted_rows == target_rows, None


def make_datum(full_tokens: list[int], weights: list[int]) -> types.Datum:
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=full_tokens[:-1]),
        loss_fn_inputs=cast(Any, {"weights": weights[1:], "target_tokens": full_tokens[1:]}),
    )


def build_example(row):
    target_sql = clean_sql_for_execution(row["sql"])
    prompt_text, full_text = render_training_texts(row["sql_prompt"], row["sql_context"], target_sql)
    prompt_tokens = tokenizer.encode(prompt_text, add_special_tokens=False)
    full_tokens = tokenizer.encode(full_text, add_special_tokens=False)
    if len(full_tokens) <= len(prompt_tokens) or len(full_tokens) > MAX_SEQ_LENGTH:
        return None

    return {
        "question": row["sql_prompt"],
        "context": row["sql_context"],
        "target": target_sql,
        "prompt_tokens": prompt_tokens,
        "active_tokens": len(full_tokens) - len(prompt_tokens),
        "datum": make_datum(full_tokens, [0] * len(prompt_tokens) + [1] * (len(full_tokens) - len(prompt_tokens))),
    }


def build_examples(dataset_split, limit, require_seed_data=False, require_target_rows=False):
    examples = []
    for row in dataset_split:
        if require_seed_data and "insert into" not in row["sql_context"].lower():
            continue
        example = build_example(row)
        if example is None:
            continue
        if require_target_rows:
            target_rows, error = run_sql(example["context"], example["target"])
            if error is not None:
                continue
            example["target_rows"] = target_rows
        examples.append(example)
        if len(examples) >= limit:
            break
    return examples


def create_sampler(alias: str):
    weights_path = training_client.save_weights_for_sampler(name=alias).result().path
    return service_client.create_sampling_client(weights_path)


async def run_eval(alias: str, examples):
    sampler = create_sampler(alias)
    execution_match = 0.0
    similarity = 0.0
    futures = [
        sampler.sample_async(
            prompt=types.ModelInput.from_ints(tokens=example["prompt_tokens"]),
            num_samples=1,
            sampling_params=types.SamplingParams(
                max_tokens=config.eval_max_tokens,
                seed=config.seed + idx,
                temperature=0.0,
            ),
        )
        for idx, example in enumerate(examples)
    ]
    responses = await asyncio.gather(*futures)

    rows = []
    for example, response in zip(examples, responses):
        predicted_sql = clean_sql_for_execution(
            tokenizer.decode(response.sequences[0].tokens if response.sequences else [], skip_special_tokens=True)
        )
        target_sql = example["target"]
        predicted = normalize_sql(predicted_sql)
        target = normalize_sql(target_sql)
        matches_execution, execution_error = sql_results_match(
            example["context"],
            predicted_sql,
            target_sql,
            target_rows=example.get("target_rows"),
        )
        rows.append(
            {
                "question": example["question"],
                "predicted_sql": predicted,
                "target_sql": target,
                "execution_match": matches_execution,
                "sqlite_error": execution_error or "",
            }
        )
        execution_match += float(matches_execution)
        similarity += SequenceMatcher(None, predicted, target).ratio()

    count = max(1, len(examples))
    return execution_match / count, similarity / count, pd.DataFrame(rows)


In [ ]:
train_examples = build_examples(split["train"], config.train_limit)
eval_examples = build_examples(
    split["test"],
    config.eval_limit,
    require_seed_data=True,
    require_target_rows=True,
)

print(f"Usable training examples: {len(train_examples):,}")
print(f"Usable eval examples: {len(eval_examples):,}")

before_execution_match, before_similarity, baseline_df = await run_eval("baseline", eval_examples)
summary_df = pd.DataFrame([
    {"metric": "Execution match", "value": f"{before_execution_match * 100:.1f}%"},
    {"metric": "Similarity", "value": f"{before_similarity * 100:.1f}%"},
    {"metric": "Eval rows", "value": len(eval_examples)},
])
display(summary_df)
display(baseline_df.head(10))


## Train the adapter

This is the same loop as the script version: `forward_backward` plus `optim_step`, with eval snapshots at the configured interval.


In [ ]:
metrics_path = Path(config.metrics_path)
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text("", encoding="utf-8")

metrics_log = []

def record_metric(step, phase, loss=None, execution_match=None, similarity=None):
    row = {"step": step, "phase": phase}
    if loss is not None:
        row["loss"] = loss
    if execution_match is not None:
        row["execution_match"] = execution_match
    if similarity is not None:
        row["similarity"] = similarity
    metrics_log.append(row)
    with metrics_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

record_metric(step=0, phase="eval", execution_match=before_execution_match, similarity=before_similarity)

losses = []
eval_history = [{"step": 0, "execution_match": before_execution_match, "similarity": before_similarity}]
last_eval_df = baseline_df

rng = random.Random(config.seed)
order = list(range(len(train_examples)))
rng.shuffle(order)
pos = 0

progress = tqdm(range(1, config.steps + 1), desc="Training", unit="step")
for step in progress:
    if pos + config.batch_size > len(order):
        rng.shuffle(order)
        pos = 0

    batch_indices = order[pos:pos + config.batch_size]
    pos += config.batch_size
    batch = [train_examples[i] for i in batch_indices]
    datums = [example["datum"] for example in batch]
    active_tokens = sum(example["active_tokens"] for example in batch)

    fwdbwd_future = await training_client.forward_backward_async(datums, "cross_entropy")
    optim_future = await training_client.optim_step_async(
        types.AdamParams(learning_rate=config.learning_rate, grad_clip_norm=config.grad_clip_norm)
    )
    fwdbwd = await fwdbwd_future
    await optim_future

    loss = float(fwdbwd.metrics.get("loss:sum", 0.0)) / max(1, active_tokens)
    losses.append(loss)
    record_metric(step=step, phase="train", loss=loss)
    progress.set_postfix(loss=f"{loss:.4f}")

    if step % config.eval_every == 0 or step == config.steps:
        execution_match, similarity, eval_df = await run_eval(f"step_{step}", eval_examples)
        last_eval_df = eval_df
        eval_history.append({
            "step": step,
            "execution_match": execution_match,
            "similarity": similarity,
        })
        record_metric(step=step, phase="eval", execution_match=execution_match, similarity=similarity)
        progress.write(
            f"[eval at step {step}] execution={execution_match * 100:.1f}% similarity={similarity * 100:.1f}%"
        )

print(f"Metrics written to {metrics_path}")


In [ ]:
eval_df = pd.DataFrame(eval_history)
plot_df = eval_df.copy()
plot_df["execution_match"] *= 100
plot_df["similarity"] *= 100

initial_loss = losses[0]
final_loss = losses[-1]
loss_drop_percent = (initial_loss - final_loss) / (abs(initial_loss) or 1.0) * 100

after_execution_match = eval_history[-1]["execution_match"]
after_similarity = eval_history[-1]["similarity"]

summary_rows = [
    {"Metric": "Execution Match (%)", "Before": f"{before_execution_match * 100:.1f}", "After": f"{after_execution_match * 100:.1f}"},
    {"Metric": "Similarity (%)", "Before": f"{before_similarity * 100:.1f}", "After": f"{after_similarity * 100:.1f}"},
    {"Metric": "Loss", "Before": f"{initial_loss:.4f}", "After": f"{final_loss:.4f}"},
    {"Metric": "Loss Drop (%)", "Before": "-", "After": f"{loss_drop_percent:.1f}"},
]
display(pd.DataFrame(summary_rows))

display(last_eval_df.head(10))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
loss_steps = list(range(1, len(losses) + 1))
axes[0].plot(loss_steps, losses, linewidth=1.0)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Step")
axes[0].grid(alpha=0.2)

axes[1].plot(plot_df["step"], plot_df["execution_match"], "o-", color="#2ca02c")
axes[1].set_title("Execution Match")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Percent")
axes[1].grid(alpha=0.2)

axes[2].plot(plot_df["step"], plot_df["similarity"], "o-", color="#ff7f0e")
axes[2].set_title("Sequence Similarity")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Percent")
axes[2].grid(alpha=0.2)

fig.suptitle(config.run_label, fontsize=13)
plt.tight_layout()
plt.savefig(config.plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {config.plot_path}")


## Playground

The current adapter is still live on the server. Edit the schema or questions below and re-run the cell to probe the trained model.


In [ ]:
PLAYGROUND_SCHEMA = """
CREATE TABLE employees (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    salary DECIMAL(10,2),
    hire_date DATE
);
""".strip()

PLAYGROUND_QUESTIONS = [
    "What are the names and salaries of employees in the Engineering department, ordered by salary descending?",
    "How many employees were hired after 2023-01-01?",
    "What is the average salary by department?",
]


def sample_sql(question, schema, sampler_name="interactive"):
    prompt_text = PLAIN_SQL_PROMPT.format(question=question, context=schema)
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    sampler = create_sampler(sampler_name)
    result = sampler.sample(
        prompt=types.ModelInput.from_ints(tokens=prompt_ids),
        num_samples=1,
        sampling_params=types.SamplingParams(max_tokens=config.eval_max_tokens, temperature=0.0),
    ).result()
    return clean_sql_for_execution(
        tokenizer.decode(result.sequences[0].tokens if result.sequences else [], skip_special_tokens=True)
    )

rows = [
    {"question": q, "generated_sql": sample_sql(q, PLAYGROUND_SCHEMA) or "-- empty --"}
    for q in PLAYGROUND_QUESTIONS
]
display(pd.DataFrame(rows))


## Next steps

- Keep `texttosql_sft_notebook.ipynb` for the Gemma 3 / chat-template flow.
- Use this notebook for the Gemma 4 E2B plain-completion path.
- If you want the scripted path instead of the notebook, run `client/texttosql_sft.py gemma4_e2b`.
- If the short run looks sane, increase `config.steps` or sweep `lora_rank` and `learning_rate`.
